# LeetCode 3142 — Check if Grid Satisfies Conditions bằng Python

Link đề: https://leetcode.com/problems/check-if-grid-satisfies-conditions/ · Độ khó: **Easy** · Chủ đề: Array, Matrix

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def satisfiesConditions(self, grid: List[List[int]]) -> bool:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: kiểm tra một ma trận số `m × n` có thỏa **hai điều kiện cho mọi ô**: (a) **bằng ô ngay bên dưới** (nếu có), (b) **khác ô ngay bên phải** (nếu có).
2. Nắm cách làm việc với **ma trận 2 chiều**: `grid[i][j]`, số hàng `len(grid)`, số cột `len(grid[0])`, **biên** (ô cuối hàng / cuối cột không có ô bên phải / bên dưới), và **chuyển vị** bằng `zip(*grid)`.
3. Đi từ cách **kiểm tra từng ô với hai điều kiện** → **nhìn theo cột: mỗi cột phải toàn một giá trị, các cột kề nhau khác giá trị** → **nhìn theo hàng: mọi hàng giống hàng đầu, hàng đầu không có hai phần tử kề bằng nhau** → **`all` / `pairwise` gọn**.
4. Hiểu mấu chốt: "bằng ô bên dưới" cho mọi ô nghĩa là **mỗi cột là một hằng số**, và "khác ô bên phải" nghĩa là **hai cột kề nhau có giá trị khác nhau**. Vậy cả lưới **được quyết định bởi hàng đầu tiên**.
5. Tránh các lỗi kinh điển: kiểm tra vượt biên (`grid[i + 1]` ở hàng cuối, `grid[i][j + 1]` ở cột cuối), **nhầm hàng với cột**, nhầm `==` / `!=`, hiểu "khác ô bên phải" thành "khác **mọi** ô trong hàng", và bỏ sót trường hợp lưới chỉ có một hàng hoặc một cột.

## 2. Đề bài & ví dụ

Cho một ma trận số nguyên `grid` kích thước `m × n`. Bạn cần kiểm tra **mỗi ô** `grid[i][j]` có thỏa:

- Nếu ô **ngay bên dưới** tồn tại (tức `i + 1 < m`) thì `grid[i][j] == grid[i + 1][j]`.
- Nếu ô **ngay bên phải** tồn tại (tức `j + 1 < n`) thì `grid[i][j] != grid[i][j + 1]`.

Trả về `true` nếu **mọi** ô đều thỏa cả hai điều kiện, ngược lại `false`.

**Ví dụ 1**

- Input: `grid = [[1, 0, 2], [1, 0, 2]]`
- Output: `true`
- Giải thích: mỗi cột có hai ô bằng nhau (`1, 1`; `0, 0`; `2, 2`) và các ô kề nhau trong hàng khác nhau (`1 ≠ 0 ≠ 2`).

**Ví dụ 2**

- Input: `grid = [[1, 1, 1], [0, 0, 0]]`
- Output: `false`
- Giải thích: ô `grid[0][0] = 1` khác ô bên dưới `grid[1][0] = 0` (vi phạm điều kiện "bằng ô bên dưới"). Ngoài ra `grid[0][0] = grid[0][1] = 1` (vi phạm điều kiện "khác ô bên phải").

**Ví dụ 3**

- Input: `grid = [[1], [2], [3]]`
- Output: `false`
- Giải thích: `grid[0][0] = 1` khác ô bên dưới `grid[1][0] = 2`.

**Ràng buộc:** `1 <= n, m <= 10`; `0 <= grid[i][j] <= 9`.

In [1]:
from typing import List
from itertools import pairwise
import random

examples = [
    ([[1, 0, 2], [1, 0, 2]], True),
    ([[1, 1, 1], [0, 0, 0]], False),
    ([[1], [2], [3]], False),
]
for grid, expected in examples:
    print("grid =", grid, "-> kỳ vọng", expected)

grid = [[1, 0, 2], [1, 0, 2]] -> kỳ vọng True
grid = [[1, 1, 1], [0, 0, 0]] -> kỳ vọng False
grid = [[1], [2], [3]] -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Ma trận: hàng, cột, ô

`grid` là list của các list. `len(grid)` là số **hàng** `m`; `len(grid[0])` là số **cột** `n`. Ô ở hàng `i`, cột `j` là `grid[i][j]`. Ô **bên dưới** là `grid[i + 1][j]` (cùng cột, hàng kế); ô **bên phải** là `grid[i][j + 1]` (cùng hàng, cột kế).

In [2]:
grid = [[1, 0, 2], [1, 0, 2]]
m, n = len(grid), len(grid[0])
print("số hàng m =", m, "| số cột n =", n)       # 2 3
print(grid[0][1], grid[1][1])                      # 0 0 : ô (0,1) và ô bên dưới (1,1)
print(grid[0][1], grid[0][2])                      # 0 2 : ô (0,1) và ô bên phải (0,2)

số hàng m = 2 | số cột n = 3
0 0
0 2


### 3.2. Biên: ô nào không có ô bên dưới / bên phải?

Ô ở **hàng cuối** (`i = m − 1`) không có ô bên dưới; ô ở **cột cuối** (`j = n − 1`) không có ô bên phải. Truy cập `grid[m][j]` hay `grid[i][n]` gây `IndexError`, nên phải kiểm tra `i + 1 < m` và `j + 1 < n` **trước** khi đọc.

In [3]:
grid = [[1, 0, 2], [1, 0, 2]]
m, n = len(grid), len(grid[0])
for i in range(m):
    for j in range(n):
        has_below = i + 1 < m
        has_right = j + 1 < n
        print(f"({i},{j}): có ô dưới? {has_below}, có ô phải? {has_right}")

(0,0): có ô dưới? True, có ô phải? True
(0,1): có ô dưới? True, có ô phải? True
(0,2): có ô dưới? True, có ô phải? False
(1,0): có ô dưới? False, có ô phải? True
(1,1): có ô dưới? False, có ô phải? True
(1,2): có ô dưới? False, có ô phải? False


### 3.3. Cột của ma trận: `zip(*grid)`

`*grid` giải nén các hàng thành các tham số riêng; `zip(*grid)` ghép các phần tử **cùng vị trí** của các hàng ⇒ cho **từng cột** (dưới dạng tuple). Đây là phép **chuyển vị** ma trận.

In [4]:
grid = [[1, 0, 2], [1, 0, 2]]
print(list(zip(*grid)))                 # [(1, 1), (0, 0), (2, 2)]
for col in zip(*grid):
    print(col, "toàn một giá trị?", len(set(col)) == 1)

[(1, 1), (0, 0), (2, 2)]
(1, 1) toàn một giá trị? True
(0, 0) toàn một giá trị? True
(2, 2) toàn một giá trị? True


### 3.4. `all`, `set` và so sánh list

- `all(điều_kiện for ...)` đúng khi mọi phần tử thỏa.
- `len(set(col)) == 1` kiểm tra mọi phần tử của `col` **bằng nhau**.
- `row == grid[0]` so sánh hai list theo từng phần tử.
- `a != b` là "khác nhau".

In [5]:
print(all(x > 0 for x in [1, 2, 3]), all(x > 1 for x in [1, 2, 3]))      # True False
print(len(set((1, 1, 1))) == 1, len(set((1, 2, 1))) == 1)                # True False
print([1, 0, 2] == [1, 0, 2], [1, 0, 2] == [1, 2, 0])                    # True False

True False
True False
True False


### 3.5. `itertools.pairwise`: các cặp phần tử kề nhau

`pairwise([1, 0, 2])` cho `(1, 0), (0, 2)`: mọi cặp phần tử **liền kề**. Rất hợp với điều kiện "khác ô bên phải" trên **một hàng**. (Cần Python `>= 3.10`; tương đương `zip(row, row[1:])`.)

In [6]:
print(list(pairwise([1, 0, 2])))                       # [(1, 0), (0, 2)]
print(all(a != b for a, b in pairwise([1, 0, 2])))     # True
print(all(a != b for a, b in pairwise([1, 1, 2])))     # False

[(1, 0), (0, 2)]
True
False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét `grid = [[1, 0, 2], [1, 0, 2]]` (`m = 2`, `n = 3`). Ta kiểm tra từng ô theo hai điều kiện:

| ô `(i, j)` | giá trị | ô dưới | cần bằng? | ô phải | cần khác? | kết quả |
|---|---|---|---|---|---|---|
| (0,0) | 1 | 1 | `1 == 1` ✔ | 0 | `1 != 0` ✔ | ✔ |
| (0,1) | 0 | 0 | `0 == 0` ✔ | 2 | `0 != 2` ✔ | ✔ |
| (0,2) | 2 | 2 | `2 == 2` ✔ | không có | — | ✔ |
| (1,0) | 1 | không có | — | 0 | `1 != 0` ✔ | ✔ |
| (1,1) | 0 | không có | — | 2 | `0 != 2` ✔ | ✔ |
| (1,2) | 2 | không có | — | không có | — | ✔ |

Mọi ô đều thỏa ⇒ `true`.

Nhận xét: điều kiện "bằng ô bên dưới" lặp lại cho mọi hàng ⇒ **mỗi cột chỉ có một giá trị** (cột `0` toàn `1`, cột `1` toàn `0`, cột `2` toàn `2`). Còn điều kiện "khác ô bên phải" chỉ còn là: **các giá trị của các cột kề nhau khác nhau** (`1, 0, 2`). Vậy **hàng đầu** đã đủ mô tả cả lưới.

In [7]:
grid = [[1, 0, 2], [1, 0, 2]]
m, n = len(grid), len(grid[0])
for j in range(n):
    col = [grid[i][j] for i in range(m)]
    print(f"cột {j}: {col} -> hằng số? {len(set(col)) == 1}")
print("các giá trị cột:", [grid[0][j] for j in range(n)])

cột 0: [1, 1] -> hằng số? True
cột 1: [0, 0] -> hằng số? True
cột 2: [2, 2] -> hằng số? True
các giá trị cột: [1, 0, 2]


## 5. Cách 1 — Brute Force / baseline: kiểm tra từng ô với hai điều kiện

**Ý tưởng.** Duyệt mọi ô `(i, j)`. Với mỗi ô: nếu có ô bên dưới thì phải **bằng**; nếu có ô bên phải thì phải **khác**. Gặp một ô vi phạm ⇒ `False` ngay; duyệt hết ⇒ `True`. Đúng từng chữ của đề.

In [8]:
def satisfies_cellwise(grid: List[List[int]]) -> bool:
    m, n = len(grid), len(grid[0])
    for i in range(m):
        for j in range(n):
            if i + 1 < m and grid[i][j] != grid[i + 1][j]:      # có ô dưới mà khác -> vi phạm
                return False
            if j + 1 < n and grid[i][j] == grid[i][j + 1]:      # có ô phải mà bằng -> vi phạm
                return False
    return True


for grid, expected in examples:
    print(satisfies_cellwise(grid), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
False (kỳ vọng False )


**Độ phức tạp (Cách 1).** Mỗi ô kiểm tra `O(1)` ⇒ **Time `O(m · n)`**, **Space `O(1)`**. Đã tối ưu về bậc (phải nhìn mọi ô trong trường hợp `true`).

**Nhận xét:** về độ phức tạp, các cách sau không nhanh hơn; chúng cho **cách nhìn khác** (theo cột / theo hàng) làm bài rõ ràng hơn.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát.** Điều kiện (a) "mọi ô bằng ô bên dưới" lan truyền: `grid[0][j] = grid[1][j] = grid[2][j] = ...`. Vậy **mỗi cột là một hằng số**, bằng `grid[0][j]`.

Điều kiện (b) "mọi ô khác ô bên phải": khi mỗi cột là hằng số `c_j`, nó trở thành `c_j != c_{j+1}` cho mọi `j`. Tức là **hàng đầu `grid[0]` không có hai phần tử kề nhau bằng nhau**.

Tóm lại: `true` ⇔ **(1) mọi hàng đều bằng hàng đầu** (mỗi cột là hằng số) **và (2) hàng đầu không có hai phần tử kề bằng nhau**. Chỉ cần so các hàng với hàng đầu và quét hàng đầu một lần.

In [9]:
grid = [[1, 0, 2], [1, 0, 2]]
rows_equal = all(row == grid[0] for row in grid)
first_row_ok = all(a != b for a, b in zip(grid[0], grid[0][1:]))
print("mọi hàng bằng hàng đầu:", rows_equal, "| hàng đầu không có hai phần tử kề bằng nhau:", first_row_ok)

mọi hàng bằng hàng đầu: True | hàng đầu không có hai phần tử kề bằng nhau: True


## 7. Các cách tiếp theo

### Cách 2 — Nhìn theo cột bằng `zip(*grid)`

**Giải thích từng đoạn.**

1. `cols = list(zip(*grid))`: danh sách các cột.
2. Mỗi cột phải toàn một giá trị: `len(set(col)) == 1`.
3. Lấy đại diện của mỗi cột `reps = [col[0] for col in cols]`.
4. Hai đại diện kề nhau phải khác nhau: `reps[j] != reps[j + 1]`.

In [10]:
def satisfies_columns(grid: List[List[int]]) -> bool:
    cols = list(zip(*grid))                                   # từng cột (tuple)
    for col in cols:
        if len(set(col)) != 1:                                # cột không phải hằng số -> vi phạm (bằng ô dưới)
            return False
    reps = [col[0] for col in cols]                           # giá trị đại diện của từng cột
    for j in range(len(reps) - 1):
        if reps[j] == reps[j + 1]:                            # hai cột kề có cùng giá trị -> vi phạm (khác ô phải)
            return False
    return True


for grid, expected in examples:
    print(satisfies_columns(grid), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
False (kỳ vọng False )


**Độ phức tạp Cách 2:** **Time `O(m · n)`**, **Space `O(m · n)`** (danh sách cột; `zip` tạo tuple cho mỗi cột).

### Cách 3 — Nhìn theo hàng: mọi hàng bằng hàng đầu, hàng đầu không có hai phần tử kề bằng nhau

**Giải thích.** Dựa vào §6. So `row == grid[0]` cho mọi hàng (mỗi cột là hằng số), rồi quét hàng đầu `grid[0]` với điều kiện "khác ô bên phải". Không tạo thêm cấu trúc (chỉ `zip` lười).

In [11]:
def satisfies_rows(grid: List[List[int]]) -> bool:
    first = grid[0]
    for row in grid:
        if row != first:                                      # có hàng khác hàng đầu -> một cột không phải hằng số
            return False
    for j in range(len(first) - 1):
        if first[j] == first[j + 1]:                          # hai ô kề trong hàng đầu bằng nhau
            return False
    return True


for grid, expected in examples:
    print(satisfies_rows(grid), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
False (kỳ vọng False )


In [12]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành Solution khi nộp)
class SolutionRows:
    def satisfiesConditions(self, grid: List[List[int]]) -> bool:
        first = grid[0]
        if any(row != first for row in grid):
            return False
        return all(first[j] != first[j + 1] for j in range(len(first) - 1))


print(SolutionRows().satisfiesConditions([[1, 0, 2], [1, 0, 2]]))      # True

True


**Độ phức tạp Cách 3:** **Time `O(m · n)`** (so mỗi hàng `O(n)`), **Space `O(1)`**.

### Cách 4 — Gọn bằng `all` và `pairwise`

**Giải thích.** Viết hai điều kiện của §6 thành hai biểu thức: `all(row == grid[0] for row in grid)` và `all(a != b for a, b in pairwise(grid[0]))`. Một dòng `return` duy nhất; đọc gần giống đề bài.

In [13]:
def satisfies_pairwise(grid: List[List[int]]) -> bool:
    return (
        all(row == grid[0] for row in grid)                        # mỗi cột là hằng số
        and all(a != b for a, b in pairwise(grid[0]))              # hàng đầu: hai ô kề khác nhau
    )


for grid, expected in examples:
    print(satisfies_pairwise(grid), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
False (kỳ vọng False )


**Độ phức tạp Cách 4:** **Time `O(m · n)`**, **Space `O(1)`** (`pairwise` là bộ sinh; `all` có đoản mạch).

## 8. TRACE từng bước

Ta trace Cách 1 (từng ô) trên `[[1, 1, 1], [0, 0, 0]]` để thấy vi phạm đầu tiên, và Cách 3 (theo hàng) trên một lưới hợp lệ.

In [14]:
def trace_cellwise(grid: List[List[int]]) -> bool:
    m, n = len(grid), len(grid[0])
    print("grid =", grid, f"(m={m}, n={n})")
    for i in range(m):
        for j in range(n):
            if i + 1 < m:
                eq = grid[i][j] == grid[i + 1][j]
                print(f"({i},{j}) vs dưới ({i + 1},{j}): {grid[i][j]} == {grid[i + 1][j]}? {eq}")
                if not eq:
                    print("  -> vi phạm 'bằng ô dưới' => False")
                    return False
            if j + 1 < n:
                ne = grid[i][j] != grid[i][j + 1]
                print(f"({i},{j}) vs phải ({i},{j + 1}): {grid[i][j]} != {grid[i][j + 1]}? {ne}")
                if not ne:
                    print("  -> vi phạm 'khác ô phải' => False")
                    return False
    print("không có vi phạm => True")
    return True


assert trace_cellwise([[1, 1, 1], [0, 0, 0]]) is False
print()
assert trace_cellwise([[1, 0, 2], [1, 0, 2]]) is True

grid = [[1, 1, 1], [0, 0, 0]] (m=2, n=3)
(0,0) vs dưới (1,0): 1 == 0? False
  -> vi phạm 'bằng ô dưới' => False

grid = [[1, 0, 2], [1, 0, 2]] (m=2, n=3)
(0,0) vs dưới (1,0): 1 == 1? True
(0,0) vs phải (0,1): 1 != 0? True
(0,1) vs dưới (1,1): 0 == 0? True
(0,1) vs phải (0,2): 0 != 2? True
(0,2) vs dưới (1,2): 2 == 2? True
(1,0) vs phải (1,1): 1 != 0? True
(1,1) vs phải (1,2): 0 != 2? True
không có vi phạm => True


In [15]:
def trace_rows(grid: List[List[int]]) -> bool:
    first = grid[0]
    print("hàng đầu =", first)
    for i, row in enumerate(grid):
        same = row == first
        print(f"hàng {i} == hàng đầu? {same}")
        if not same:
            return False
    for j in range(len(first) - 1):
        diff = first[j] != first[j + 1]
        print(f"hàng đầu: first[{j}]={first[j]} != first[{j + 1}]={first[j + 1]}? {diff}")
        if not diff:
            return False
    return True


assert trace_rows([[1, 0, 2], [1, 0, 2]]) is True

hàng đầu = [1, 0, 2]
hàng 0 == hàng đầu? True
hàng 1 == hàng đầu? True
hàng đầu: first[0]=1 != first[1]=0? True
hàng đầu: first[1]=0 != first[2]=2? True


Bảng trace Cách 1 với `[[1, 1, 1], [0, 0, 0]]`:

| ô `(i, j)` | ô dưới | `==`? | ô phải | `!=`? | kết quả |
|---|---|---|---|---|---|
| (0,0) | `(1,0)` = 0 | `1 == 0`? **không** | — | — | **vi phạm ⇒ `False` ngay** |

Ô đầu tiên đã sai nên dừng ở đó. Với lưới hợp lệ `[[1,0,2],[1,0,2]]` thì mọi ô qua được, kết quả `True`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 1 (kiểm tra từng ô): bám sát đề, đúng cả khi lưới chỉ có một hàng/một cột, `O(m · n)`.

In [16]:
from typing import List

class Solution:
    def satisfiesConditions(self, grid: List[List[int]]) -> bool:
        m, n = len(grid), len(grid[0])
        for i in range(m):
            for j in range(n):
                if i + 1 < m and grid[i][j] != grid[i + 1][j]:      # có ô dưới nhưng khác -> sai
                    return False
                if j + 1 < n and grid[i][j] == grid[i][j + 1]:      # có ô phải nhưng bằng -> sai
                    return False
        return True


sol = Solution()
for grid, expected in examples:
    print(grid, "->", sol.satisfiesConditions(grid))

[[1, 0, 2], [1, 0, 2]] -> True
[[1, 1, 1], [0, 0, 0]] -> False
[[1], [2], [3]] -> False


## 10. Vì sao thuật toán đúng

- **Đúng định nghĩa:** thuật toán kiểm tra đúng hai điều kiện của đề cho **từng** ô; điều kiện chỉ áp dụng khi ô lân cận **tồn tại** (`i + 1 < m`, `j + 1 < n`), nên ta không đọc ngoài biên. Có vi phạm ⇒ trả `False`; không ô nào vi phạm ⇒ `True`.
- **Các cách nhìn theo cột/hàng tương đương:** (a) áp dụng liên tiếp cho các hàng `0, 1, 2, ...` cho `grid[0][j] = grid[1][j] = ...` nên mỗi cột là hằng số; đảo lại, nếu mọi cột là hằng số thì (a) đúng. (b) với cột là hằng số, `grid[i][j] != grid[i][j+1]` tương đương `grid[0][j] != grid[0][j+1]` với mọi `i`. Vậy `true` ⇔ mọi hàng bằng hàng đầu và hàng đầu không có hai ô kề bằng nhau.
- **Biên:** lưới một hàng (`m = 1`): điều kiện (a) không có ô nào áp dụng, chỉ còn (b) trên hàng đó. Lưới một cột (`n = 1`): chỉ còn (a): mọi phần tử của cột bằng nhau.

## 11. Phân tích Time / Space Complexity

Gọi `m` là số hàng, `n` là số cột.

| Cách | Time | Space |
|---|---|---|
| 1. Từng ô | `O(m · n)` | `O(1)` |
| 2. Theo cột (`zip(*grid)`) | `O(m · n)` | `O(m · n)` |
| 3. Theo hàng | `O(m · n)` | `O(1)` |
| 4. `all` + `pairwise` | `O(m · n)` | `O(1)` |

Với `m, n <= 10` tất cả đều tức thì. Trường hợp xấu nhất `true` phải đọc mọi ô nên `O(m · n)` là tối ưu.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [17]:
import copy

sol = Solution()
impls = {
    "cellwise": satisfies_cellwise,
    "columns": satisfies_columns,
    "rows": satisfies_rows,
    "pairwise": satisfies_pairwise,
    "leetcode": sol.satisfiesConditions,
}


def check_all(grid, expected, label):
    for key, fn in impls.items():
        g = copy.deepcopy(grid)
        actual = fn(g)
        assert actual == expected, (label, key, grid, actual, expected)
        assert g == grid, ("input bị sửa", key)
    print("PASS", label)


for idx, (grid, expected) in enumerate(examples, 1):
    check_all(grid, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [18]:
check_all([[5]], True, "một ô: không có ô dưới/phải")
check_all([[1, 2]], True, "một hàng, hai ô khác nhau")
check_all([[1, 1]], False, "một hàng, hai ô bằng nhau")
check_all([[1, 2, 1]], True, "một hàng: chỉ cần kề nhau khác nhau (1 và 1 không kề)")
check_all([[3], [3]], True, "một cột, hai ô bằng nhau")
check_all([[3], [4]], False, "một cột, hai ô khác nhau")
check_all([[3], [3], [3], [3]], True, "một cột toàn bằng nhau")
check_all([[0, 9], [0, 9]], True, "hai cột kề khác nhau")
check_all([[0, 0], [0, 0]], False, "mọi ô bằng nhau: vi phạm 'khác ô phải'")
check_all([[1, 2, 1], [1, 2, 1]], True, "cột 0 và cột 2 cùng giá trị nhưng không kề")
check_all([[1, 2], [1, 3]], False, "cột 1 không phải hằng số")
check_all([[1, 2], [2, 1]], False, "hai hàng khác nhau")
check_all([[1, 2, 3]] * 10, True, "m = 10 hàng giống nhau")
check_all([[j % 2 for j in range(10)] for _ in range(10)], True, "10x10: 0,1,0,1,... mỗi cột hằng số")
check_all([[1] * 10 for _ in range(10)], False, "10x10 toàn 1")

PASS một ô: không có ô dưới/phải
PASS một hàng, hai ô khác nhau
PASS một hàng, hai ô bằng nhau
PASS một hàng: chỉ cần kề nhau khác nhau (1 và 1 không kề)
PASS một cột, hai ô bằng nhau
PASS một cột, hai ô khác nhau
PASS một cột toàn bằng nhau
PASS hai cột kề khác nhau
PASS mọi ô bằng nhau: vi phạm 'khác ô phải'
PASS cột 0 và cột 2 cùng giá trị nhưng không kề
PASS cột 1 không phải hằng số
PASS hai hàng khác nhau
PASS m = 10 hàng giống nhau
PASS 10x10: 0,1,0,1,... mỗi cột hằng số
PASS 10x10 toàn 1


### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi phổ biến: (a) **chỉ kiểm tra điều kiện theo hàng** (quên "bằng ô dưới"); (b) **chỉ kiểm tra điều kiện theo cột** (quên "khác ô phải"); (c) **hiểu "khác ô bên phải" thành "mọi ô trong hàng đều khác nhau"**.

In [19]:
def wrong_only_horizontal(grid):
    return all(a != b for row in grid for a, b in zip(row, row[1:]))          # SAI: thiếu 'bằng ô dưới'


def wrong_only_vertical(grid):
    return all(row == grid[0] for row in grid)                                 # SAI: thiếu 'khác ô phải'


def wrong_all_distinct_in_row(grid):
    return all(row == grid[0] for row in grid) and len(set(grid[0])) == len(grid[0])      # SAI: quá chặt


print(wrong_only_horizontal([[1, 2], [3, 4]]), "!=", satisfies_cellwise([[1, 2], [3, 4]]))
print(wrong_only_vertical([[1, 1], [1, 1]]), "!=", satisfies_cellwise([[1, 1], [1, 1]]))
print(wrong_all_distinct_in_row([[1, 2, 1], [1, 2, 1]]), "!=", satisfies_cellwise([[1, 2, 1], [1, 2, 1]]))
assert wrong_only_horizontal([[1, 2], [3, 4]]) != satisfies_cellwise([[1, 2], [3, 4]])
assert wrong_only_vertical([[1, 1], [1, 1]]) != satisfies_cellwise([[1, 1], [1, 1]])
assert wrong_all_distinct_in_row([[1, 2, 1], [1, 2, 1]]) != satisfies_cellwise([[1, 2, 1], [1, 2, 1]])

True != False
True != False
False != True


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Nếu sinh lưới hoàn toàn ngẫu nhiên thì gần như luôn `false`; để cân bằng, ta sinh **lưới hợp lệ** (cột là hằng số, các cột kề khác nhau) rồi **gây nhiễu** một vài ô. Ngoài ra vét cạn **mọi lưới `2 × 3` trên giá trị `{0, 1, 2}`** (`3^6 = 729` lưới).

In [20]:
import itertools


def random_valid_grid(rng, m, n, hi):
    cols = []
    for j in range(n):
        while True:
            v = rng.randint(0, hi)
            if not cols or v != cols[-1]:
                cols.append(v)
                break
    return [list(cols) for _ in range(m)]


rng = random.Random(3142)
true_count = false_count = 0
for trial in range(4000):
    m, n = rng.randint(1, 10), rng.randint(1, 10)
    hi = rng.choice([2, 3, 9])
    grid = random_valid_grid(rng, m, n, hi)
    for _ in range(rng.choice([0, 0, 1, 2])):                     # gây nhiễu 0..2 ô
        grid[rng.randrange(m)][rng.randrange(n)] = rng.randint(0, 9)
    assert 1 <= m <= 10 and 1 <= n <= 10 and all(0 <= v <= 9 for row in grid for v in row)
    ref = satisfies_cellwise(copy.deepcopy(grid))
    true_count += ref
    false_count += not ref
    for key, fn in impls.items():
        assert fn(copy.deepcopy(grid)) == ref, (key, grid)
print("PASS 4000 lưới ngẫu nhiên: True =", true_count, ", False =", false_count)
assert true_count > 300 and false_count > 300

checked = 0
for bits in itertools.product(range(3), repeat=6):
    grid = [list(bits[0:3]), list(bits[3:6])]
    ref = satisfies_cellwise(copy.deepcopy(grid))
    for key, fn in impls.items():
        assert fn(copy.deepcopy(grid)) == ref, (key, grid)
    checked += 1
print("PASS vét cạn", checked, "lưới 2 x 3")

PASS 4000 lưới ngẫu nhiên: True = 2173 , False = 1827
PASS vét cạn 729 lưới 2 x 3


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Vượt biên khi đọc ô dưới / ô phải

In [21]:
def sai_vuot_bien(grid):
    m, n = len(grid), len(grid[0])
    for i in range(m):
        for j in range(n):
            if grid[i][j] != grid[i + 1][j]:             # SAI: i = m - 1 thì grid[m] không tồn tại
                return False
            if grid[i][j] == grid[i][j + 1]:             # SAI: j = n - 1 thì grid[i][n] không tồn tại
                return False
    return True


try:
    sai_vuot_bien([[1, 0, 2], [1, 0, 2]])
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().satisfiesConditions([[1, 0, 2], [1, 0, 2]]))

IndexError: list index out of range
đúng: True


**Vì sao sai:** ô ở hàng cuối không có ô dưới; ô ở cột cuối không có ô phải. Đề chỉ yêu cầu điều kiện **khi ô đó tồn tại**. **Cách sửa:** `i + 1 < m and ...`, `j + 1 < n and ...` (kiểm tra biên **trước**, nhờ đoản mạch của `and`).

### Lỗi 2: Nhầm `==` và `!=` (đảo hai điều kiện)

In [22]:
def sai_dao_dieu_kien(grid):
    m, n = len(grid), len(grid[0])
    for i in range(m):
        for j in range(n):
            if i + 1 < m and grid[i][j] == grid[i + 1][j]:      # SAI: đảo -> phải là != để báo vi phạm
                return False
            if j + 1 < n and grid[i][j] != grid[i][j + 1]:      # SAI: đảo
                return False
    return True


print("sai:", sai_dao_dieu_kien([[1, 0, 2], [1, 0, 2]]), "| đúng:", Solution().satisfiesConditions([[1, 0, 2], [1, 0, 2]]))

sai: False | đúng: True


**Vì sao sai:** ta đang **tìm vi phạm**: vi phạm của "bằng ô dưới" là **khác** ô dưới (`!=`), vi phạm của "khác ô phải" là **bằng** ô phải (`==`). Đảo cả hai cho kết quả ngược. **Cách sửa:** viết rõ trong comment "điều kiện đề" và "điều kiện vi phạm" trước khi viết `if`.

### Lỗi 3: Nhầm hàng với cột (`m` và `n`)

In [23]:
# lỗi: đảo m và n khi lưới không vuông
def sai_nham_m_n(grid):
    m, n = len(grid[0]), len(grid)
    for i in range(m):
        for j in range(n):
            if i + 1 < m and grid[i][j] != grid[i + 1][j]:
                return False
            if j + 1 < n and grid[i][j] == grid[i][j + 1]:
                return False
    return True


try:
    print("sai:", sai_nham_m_n([[1, 0, 2], [1, 0, 2]]))
except IndexError as e:
    print("sai -> IndexError:", e)
print("đúng:", Solution().satisfiesConditions([[1, 0, 2], [1, 0, 2]]))

sai -> IndexError: list index out of range
đúng: True


**Vì sao sai:** số **hàng** là `m = len(grid)`; số **cột** là `n = len(grid[0])`. Đảo hai số làm chỉ số hàng chạy tới số cột (3 > 2 hàng), gây `IndexError` hoặc kiểm tra sai ô. **Cách sửa:** `m, n = len(grid), len(grid[0])` và nhớ "hàng trước, cột sau".

### Lỗi 4: Hiểu "khác ô bên phải" thành "mọi ô trong hàng đều khác nhau"

In [24]:
def sai_qua_chat(grid):
    first = grid[0]
    return all(row == first for row in grid) and len(set(first)) == len(first)     # SAI: yêu cầu mọi giá trị khác nhau


print("sai:", sai_qua_chat([[1, 2, 1], [1, 2, 1]]), "| đúng:", Solution().satisfiesConditions([[1, 2, 1], [1, 2, 1]]))

sai: False | đúng: True


**Vì sao sai:** đề chỉ cần ô **kề nhau** khác nhau (`1 ≠ 2`, `2 ≠ 1`); hai ô `1` ở hai đầu (không kề) được phép bằng nhau. **Cách sửa:** so sánh `first[j] != first[j + 1]` (cặp kề), không dùng `set`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài ma trận với điều kiện **"bằng ô bên dưới" cho mọi ô** ⇒ nghĩ ngay "**mỗi cột là hằng số**" ⇒ chỉ cần xét **một hàng**.
- Điều kiện "khác ô lân cận" ⇒ quét `pairwise` (hoặc `zip(row, row[1:])`).
- Điều kiện chỉ áp dụng **khi ô lân cận tồn tại** ⇒ kiểm tra biên trước bằng `i + 1 < m and ...`.
- Nhớ `m = len(grid)` (hàng), `n = len(grid[0])` (cột); `zip(*grid)` cho cột.
- Lưới một hàng/một cột là trường hợp biên quan trọng — luôn thử `[[5]]`, `[[1, 2]]`, `[[3], [3]]`.

## 15. Biến thể của bài

1. **Mọi ô bằng ô bên dưới và bằng ô bên phải** (lưới toàn một giá trị).
2. **Đếm số ô vi phạm** thay vì chỉ `true/false`.
3. **Số thao tác ít nhất** để đổi các giá trị khiến lưới thỏa điều kiện (xem *Minimum Number of Operations to Satisfy Conditions (3122)* — quy hoạch động theo cột).
4. **Cột kề nhau khác nhau, hàng kề nhau bằng nhau** (đổi vai trò).
5. **Toeplitz Matrix (766)**, **Valid Sudoku (36)**.

In [25]:
def count_violations(grid):
    m, n = len(grid), len(grid[0])
    total = 0
    for i in range(m):
        for j in range(n):
            if i + 1 < m and grid[i][j] != grid[i + 1][j]:
                total += 1
            if j + 1 < n and grid[i][j] == grid[i][j + 1]:
                total += 1
    return total


def all_equal_grid(grid):
    return len({v for row in grid for v in row}) == 1


print(count_violations([[1, 1, 1], [0, 0, 0]]))      # 3 vertical + 2 horizontal ở hàng 0 + 2 ở hàng 1 = 7
print(count_violations([[1, 0, 2], [1, 0, 2]]))      # 0
assert count_violations([[1, 1, 1], [0, 0, 0]]) == 3 + 2 + 2
assert count_violations([[1, 0, 2], [1, 0, 2]]) == 0
assert all_equal_grid([[2, 2], [2, 2]]) and not all_equal_grid([[2, 3]])
for grid, expected in examples:
    assert (count_violations(grid) == 0) == expected

7
0


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Từng ô | hai điều kiện cho mỗi ô | `O(m · n)` | `O(1)` | **nên nộp**, bám sát đề |
| 2. Theo cột | `zip(*grid)`, cột hằng số, đại diện khác nhau | `O(m · n)` | `O(m · n)` | khi nghĩ theo cột |
| 3. Theo hàng | mọi hàng bằng hàng đầu + hàng đầu kề khác | `O(m · n)` | `O(1)` | khi đã rút ra tính chất |
| 4. `all` + `pairwise` | gọn một dòng | `O(m · n)` | `O(1)` | khi thích viết ngắn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(zip(*[[1, 2], [3, 4]]))`, `list(pairwise([5, 5, 6]))`, `[[7, 8], [7, 8]] == [[7, 8], [7, 9]]`, `Solution().satisfiesConditions([[4, 4], [4, 4]])`, `Solution().satisfiesConditions([[4], [4], [4]])`.

**Bài 2 (tính tay).** Với `grid = [[2, 5, 2], [2, 5, 2], [2, 5, 2]]` và `grid = [[2, 5, 2], [2, 6, 2], [2, 5, 2]]`: kiểm tra từng cột và từng cặp ô kề trong hàng đầu, cho kết quả.

**Bài 3 (điền TODO).** Hoàn thành `satisfies_todo(grid)` theo Cách 3 (theo hàng).

In [26]:
def satisfies_todo(grid):
    first = grid[0]
    # TODO 1: nếu có hàng nào khác first thì trả False
    # TODO 2: duyệt j từ 0 tới len(first) - 2: nếu first[j] == first[j + 1] thì trả False
    # TODO 3: nếu không vi phạm thì trả True
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `satisfies_columns2(grid)` theo kiểu **theo cột** nhưng **không dùng `zip(*grid)`**: dùng hai vòng lặp `for j in range(n)` và `for i in range(m)` để tạo cột.

**Bài 5 (biến thể / mini challenge).** (a) Viết `first_violation(grid)` trả về **ô `(i, j)` đầu tiên** (theo thứ tự hàng rồi cột) bị vi phạm bất kỳ điều kiện nào, hoặc `None`. (b) Viết `min_changes_to_satisfy(grid)` trả về **số ô ít nhất** phải đổi giá trị để lưới thỏa điều kiện — gợi ý: bài khó hơn (3122) cần quy hoạch động; ở đây giới hạn giá trị `0..2` và `m, n <= 3`, hãy **vét cạn** mọi lưới đích hợp lệ.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[(1, 3), (2, 4)]`, `[(5, 5), (5, 6)]`, `False`, `False` (hai ô kề bằng nhau), `True`. Bài 3: ba chỗ TODO tương ứng ba bước ở Cách 3.
- **Mức 2 (vừa):** Bài 2: lưới đầu: cột `(2,2,2)`, `(5,5,5)`, `(2,2,2)` hằng số; hàng đầu `2, 5, 2`: `2 != 5`, `5 != 2` ⇒ `True`. Lưới sau: cột giữa `(5, 6, 5)` không hằng số ⇒ `False`. Bài 4: `col = [grid[i][j] for i in range(m)]`.
- **Mức 3 (sâu):** Bài 5(a): thêm `i + 1 < m` / `j + 1 < n` và trả `(i, j)` khi vi phạm. 5(b): sinh mọi lưới đích `m × n` trên `{0, 1, 2}` (itertools.product), lọc lưới thỏa điều kiện, đếm số ô khác nhau so với lưới gốc, lấy nhỏ nhất.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert list(zip(*[[1, 2], [3, 4]])) == [(1, 3), (2, 4)]
assert list(pairwise([5, 5, 6])) == [(5, 5), (5, 6)]
assert ([[7, 8], [7, 8]] == [[7, 8], [7, 9]]) is False
assert Solution().satisfiesConditions([[4, 4], [4, 4]]) is False
assert Solution().satisfiesConditions([[4], [4], [4]]) is True

# Bài 2
assert Solution().satisfiesConditions([[2, 5, 2], [2, 5, 2], [2, 5, 2]]) is True
assert Solution().satisfiesConditions([[2, 5, 2], [2, 6, 2], [2, 5, 2]]) is False


# Bài 3
def satisfies_todo_ans(grid):
    first = grid[0]
    for row in grid:
        if row != first:
            return False
    for j in range(len(first) - 1):
        if first[j] == first[j + 1]:
            return False
    return True


for grid, expected in examples:
    assert satisfies_todo_ans(grid) == expected


# Bài 4
def satisfies_columns2(grid):
    m, n = len(grid), len(grid[0])
    reps = []
    for j in range(n):
        col = [grid[i][j] for i in range(m)]
        if len(set(col)) != 1:
            return False
        reps.append(col[0])
    return all(reps[j] != reps[j + 1] for j in range(n - 1))


for grid, expected in examples:
    assert satisfies_columns2(grid) == expected


# Bài 5(a)
def first_violation(grid):
    m, n = len(grid), len(grid[0])
    for i in range(m):
        for j in range(n):
            if i + 1 < m and grid[i][j] != grid[i + 1][j]:
                return (i, j)
            if j + 1 < n and grid[i][j] == grid[i][j + 1]:
                return (i, j)
    return None


assert first_violation([[1, 1, 1], [0, 0, 0]]) == (0, 0)
assert first_violation([[1, 0, 2], [1, 0, 2]]) is None
assert first_violation([[1, 0, 0], [1, 0, 0]]) == (0, 1)


# Bài 5(b): vét cạn trên {0,1,2}, m, n <= 3
def min_changes_to_satisfy(grid):
    m, n = len(grid), len(grid[0])
    best = m * n
    for cols in itertools.product(range(3), repeat=n):          # mỗi cột một giá trị
        if any(cols[j] == cols[j + 1] for j in range(n - 1)):
            continue
        target = [list(cols) for _ in range(m)]
        diff = sum(grid[i][j] != target[i][j] for i in range(m) for j in range(n))
        best = min(best, diff)
    return best


assert min_changes_to_satisfy([[1, 0, 2], [1, 0, 2]]) == 0
assert min_changes_to_satisfy([[1, 1, 1], [0, 0, 0]]) == 3
assert min_changes_to_satisfy([[0]]) == 0
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (kiểm tra điều kiện giữa ô và các ô lân cận, có kiểm tra biên)

```python
from typing import List

class Solution:
    def satisfiesConditions(self, grid: List[List[int]]) -> bool:
        m, n = len(grid), len(grid[0])
        for i in range(m):
            for j in range(n):
                if i + 1 < m and grid[i][j] != grid[i + 1][j]:
                    return False
                if j + 1 < n and grid[i][j] == grid[i][j + 1]:
                    return False
        return True
```

**Checklist**

- [ ] `m = len(grid)` (hàng), `n = len(grid[0])` (cột): không nhầm?
- [ ] Kiểm tra biên `i + 1 < m`, `j + 1 < n` **trước** khi đọc ô lân cận?
- [ ] Vi phạm của "bằng ô dưới" là `!=`; vi phạm của "khác ô phải" là `==`?
- [ ] Chỉ so ô **kề nhau** (không yêu cầu mọi ô trong hàng khác nhau)?
- [ ] Đã thử `[[5]]`, `[[1, 1]]`, `[[3], [4]]`, và lưới hợp lệ có giá trị lặp ở cột không kề?
- [ ] Skeleton đúng `class Solution` / `def satisfiesConditions(self, grid: List[List[int]]) -> bool`?